In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# COMMAND ----------

from pyspark.sql.functions import (
    col, count, countDistinct, trim, length,
    min as spark_min, max as spark_max,
    sum as spark_sum, to_timestamp
)

RAW_TABLE_NAME = "physical_lojas.csv"

RAW_PATH = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    f"batch-data/{RAW_TABLE_NAME}/"
)

adls_options = get_adls_options()

print(f"Lendo Raw: {RAW_TABLE_NAME}")
print(f"Caminho: {RAW_PATH}")

In [0]:
# COMMAND ----------

def read_raw_csv(path):
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .option("sep", ",")
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .options(**adls_options)
        .load(path)
    )

    # Caso venha separado por ponto e vírgula
    if len(df.columns) == 1:
        df = (
            spark.read
            .format("csv")
            .option("header", True)
            .option("inferSchema", False)
            .option("sep", ";")
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .options(**adls_options)
            .load(path)
        )

    return df

df_lojas_raw = read_raw_csv(RAW_PATH)

print("Leitura concluída.")
print(f"Total de colunas: {len(df_lojas_raw.columns)}")

df_lojas_raw.printSchema()
display(df_lojas_raw.limit(40))